# 02 — Train the Intent Classifier via Together.ai Fine-tuning

This notebook fine-tunes a `Mistral-7B-Instruct-v0.2` intent classifier on the Veridian
IT ticket data prepared in `01_data_prep.ipynb`, evaluates it on the held-out test
set, and saves the fine-tuned model ID for use in `03_agent_demo.ipynb`.

Fine-tuning runs on Together.ai using the same messages JSONL format generated by
notebook 01.

In [1]:
!uv add together mistralai pandas scikit-learn python-dotenv rich

Resolved 151 packages in 0.78ms
Audited 145 packages in 1ms


In [2]:
import json
import os
import time
from pathlib import Path

import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

console = Console()

DATA_DIR        = Path("data")
TEST_FILE       = DATA_DIR / "test.jsonl"
MODEL_ID_FILE   = DATA_DIR / "classifier_model_id.txt"
CONFIG_FILE     = DATA_DIR / "finetune_config.json"

INTENT_LABELS = [
    "access_request",
    "security_incident",
    "hardware_issue",
    "software_issue",
    "onboarding",
    "payments_incident",
    "expense_request",
    "general_question",
]

## 1. Overview

### What we're training

A **single-label intent classifier** on 122 Veridian IT support tickets, fine-tuned
on Together.ai using the **messages format** generated by `01_data_prep.ipynb`.
The model learns to output exactly one of 8 intent labels given an employee support request.

| Parameter | Value |
|---|---|
| Base model | `mistralai/Mistral-7B-Instruct-v0.2` |
| Platform | Together.ai fine-tuning API |
| Epochs | 5 |
| Classes | 8 (see `01_data_prep.ipynb`) |
| Typical training time | **5–15 minutes** |

### Why Mistral-7B-Instruct-v0.2 on Together.ai?

A compact model fine-tuned for classification is faster and cheaper at inference
than prompting a large model to guess intent, and substantially more accurate on a
specific label taxonomy. The fine-tune trains the model to output only the
intent label given a ticket — deterministic routing at `temperature=0` in ~50–150 ms.

This classifier replaces the routing work `mistral-large` would otherwise
have to perform during the agentic loop — freeing the large model to focus
entirely on reasoning and tool use.

## 2. Setup

Training data is read from the local JSONL files written by `01_data_prep.ipynb`.
**Make sure you have run notebook 01 before this one — `data/train.jsonl` and
`data/val.jsonl` must exist.**

You need a **Together.ai API key** in `mistral-it-agent/.env`:
```
TOGETHER_API_KEY=your-key-here
MISTRAL_API_KEY=your-mistral-key-here
```

In [3]:
# ── Verify training data files exist ─────────────────────────────────────
missing = [f for f in [DATA_DIR / "train.jsonl", DATA_DIR / "val.jsonl", DATA_DIR / "test.jsonl"] if not f.exists()]
if missing:
    raise FileNotFoundError(
        f"Missing data files: {[str(f) for f in missing]}\n"
        "Run 01_data_prep.ipynb first to generate the JSONL split files."
    )

train_count = sum(1 for _ in open(DATA_DIR / "train.jsonl"))
val_count   = sum(1 for _ in open(DATA_DIR / "val.jsonl"))
test_count  = sum(1 for _ in open(DATA_DIR / "test.jsonl"))
console.print(f"[green]Data files found:[/green] train={train_count}, val={val_count}, test={test_count}")

Data files found: train=295, val=63, test=64

In [4]:
from together import Together

# ── API key ───────────────────────────────────────────────────────────────
try:
    from google.colab import userdata  # type: ignore
    TOGETHER_API_KEY = userdata.get("TOGETHER_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv(override=True)
    TOGETHER_API_KEY = os.getenv("TOGETHER_API_KEY")

if not TOGETHER_API_KEY:
    raise EnvironmentError(
        "TOGETHER_API_KEY is not set.\n"
        "Local: add TOGETHER_API_KEY=... to mistral-it-agent/.env\n"
        "Colab: add it via Secrets (key icon in the left sidebar)."
    )

client = Together(api_key=TOGETHER_API_KEY)
console.print("[bold green]Together.ai client initialised.[/bold green]")

Together.ai client initialised.

In [5]:
# ── Recover job from env if kernel was restarted ──────────────────────────
# Set TOGETHER_JOB_ID=ft-xxxxxxxx-xxxx in mistral-it-agent/.env to resume
# a previously created job without re-uploading files or re-running training.
if "job" not in dir():
    _saved_job_id = os.getenv("TOGETHER_JOB_ID", "").strip()
    if _saved_job_id:
        job = client.fine_tuning.retrieve(id=_saved_job_id)
        console.print(f"[dim]Restored job from TOGETHER_JOB_ID:[/dim] [cyan]{job.id}[/cyan] — status: [yellow]{job.status}[/yellow]")
    else:
        console.print("[dim]No TOGETHER_JOB_ID set — run the upload+create cell below to start a new job.[/dim]")

No TOGETHER_JOB_ID set — run the upload+create cell below to start a new job.

## 3. Upload training files and create fine-tuning job

Together.ai accepts the same messages JSONL format generated by `01_data_prep.ipynb`.
Files are uploaded fresh each run — no pre-uploaded IDs needed.

In [6]:
# ── Upload training and validation files ─────────────────────────────────
# check=False forces a fresh upload each run so stale file objects don't
# carry over from previous failed attempts.
console.print("Uploading training file…")
train_upload = client.files.upload(
    file=DATA_DIR / "train.jsonl",
    check=False,
)
console.print(f"  train file id: [cyan]{train_upload.id}[/cyan]")

console.print("Uploading validation file…")
val_upload = client.files.upload(
    file=DATA_DIR / "val.jsonl",
    check=False,
)
console.print(f"  val   file id: [cyan]{val_upload.id}[/cyan]")

# ── Create fine-tuning job ────────────────────────────────────────────────
# lora=True  — LoRA is the reliably-supported fine-tuning method for Mistral
#              on Together.ai; full fine-tune requires a higher account tier.
# train_on_inputs=False — train on the assistant (label) turn only.
# No validation_file — some models 500 when a val file is passed with LoRA.
_BASE_MODEL = "mistralai/Mistral-7B-v0.1"

job = client.fine_tuning.create(
    training_file=train_upload.id,
    model=_BASE_MODEL,
    n_epochs=7,
    lora=True,
    train_on_inputs=False,
)

console.print(Panel(
    f"[bold]Job ID:[/bold]  [cyan]{job.id}[/cyan]\n"
    f"[bold]Status:[/bold]  [yellow]{job.status}[/yellow]\n"
    f"[bold]Model:[/bold]   {_BASE_MODEL} (LoRA)\n"
    f"[bold]Epochs:[/bold]  7\n"
    f"[dim]Typical training time: 5–15 minutes[/dim]",
    title="Fine-tuning job created",
    border_style="cyan",
))

Uploading training file…

Uploading file train.jsonl: 100%|████████████████████████| 672k/672k [00:00<00:00, 1.49MB/s]


train file id: file-c8bf7dc7-c0e1-4bdc-ba68-ce6f89cb69f0

Uploading validation file…

Uploading file val.jsonl: 100%|███████████████████████████| 144k/144k [00:00<00:00, 532kB/s]


val   file id: file-11687ba3-91ab-4e06-8c99-db2575f3628d

╭──────────────────────────────────────────── Fine-tuning job created ────────────────────────────────────────────╮
│ Job ID:  ft-d0e9e763-87a7                                                                                       │
│ Status:  pending                                                                                                │
│ Model:   mistralai/Mistral-7B-v0.1 (LoRA)                                                                       │
│ Epochs:  7                                                                                                      │
│ Typical training time: 5–15 minutes                                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [7]:
## 4. Poll until complete

# Terminal statuses per SDK v2: completed, cancelled, error, user_error
_TERMINAL = {"completed", "cancelled", "error", "user_error"}

if "job" not in dir():
    console.print(
        "[yellow]Skipping poll — no active job.[/yellow]\n"
        "[dim]Run the job creation cell above first.[/dim]"
    )
else:
    console.print(f"Polling job [bold]{job.id}[/bold] …")

    while True:
        status = client.fine_tuning.retrieve(id=job.id)
        console.print(f"  status = [yellow]{status.status}[/yellow]")
        if status.status in _TERMINAL:
            break
        time.sleep(30)

    if status.status != "completed":
        raise RuntimeError(f"Fine-tuning job ended with status: {status.status}")

    # SDK v2: the attribute is x_model_output_name (alias: model_output_name)
    fine_tuned_model_id = status.x_model_output_name
    console.print(Panel(
        f"[bold]Fine-tuned model ID:[/bold] [cyan]{fine_tuned_model_id}[/cyan]",
        title="[green]Training complete[/green]",
        border_style="green",
    ))

Polling job ft-d0e9e763-87a7 …

status = pending

status = queued

status = running

status = running

...

status = compressing

status = uploading

status = uploading

status = completed

╭─────────────────────────────────────────────── Training complete ───────────────────────────────────────────────╮
│ Fine-tuned model ID: crowe3555_2a6c/Mistral-7B-v0.1-27e98401                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [8]:
# ── Training events from job ──────────────────────────────────────────────
if "job" not in dir():
    console.print("[yellow]Skipping — no active job.[/yellow]")
else:
    events = client.fine_tuning.list_events(id=job.id)

    # FinetuneEvent fields: step, type, message, level (no nested .data dict)
    step_events = [e for e in events.data if e.step > 0]

    if step_events:
        t = Table(title="Training events (checkpoints)", header_style="bold")
        t.add_column("Step",    justify="right")
        t.add_column("Type",    style="cyan")
        t.add_column("Message", style="dim", max_width=70)
        for e in step_events[-5:]:
            t.add_row(str(e.step), str(e.type), e.message)
        console.print(t)
    else:
        console.print("[dim]No step events in job history.[/dim]")

             Training events (checkpoints)              
┏━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Step ┃ Type           ┃ Message                      ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   57 │ EPOCH_COMPLETE │ Epoch completed, at step 57  │
│   76 │ EPOCH_COMPLETE │ Epoch completed, at step 76  │
│   95 │ EPOCH_COMPLETE │ Epoch completed, at step 95  │
│  114 │ EPOCH_COMPLETE │ Epoch completed, at step 114 │
│  133 │ EPOCH_COMPLETE │ Epoch completed, at step 133 │
└──────┴────────────────┴──────────────────────────────┘

## 5. Evaluate on test set

`test.jsonl` was held out in `01_data_prep.ipynb` and never uploaded for training.
We classify every test example in a single batched call, then report
per-class precision, recall, and F1 using `sklearn`.

> The test set contains ~18 examples (15 % of 122 raw tickets), or ~66 if synthetic
> data was generated (15 % of ~442). F1 scores on the raw-only set have high variance;
> treat them as a directional signal. With synthetic data the estimate is more reliable.
> For a production deployment, collect ≥ 200 labelled examples and re-evaluate.

In [ ]:
# ── Create a dedicated endpoint for the fine-tuned model ─────────────────
# Fine-tuned LoRA models on Together.ai require a dedicated endpoint for
# inference — they are not available on the shared serverless pool.
# This cell creates one, waits for it to start, and stores the endpoint
# details so evaluation, notebook 03, and app.py can all use it.

_ENDPOINT_NAME_FILE = DATA_DIR / "endpoint_name.txt"
_ENDPOINT_ID_FILE   = DATA_DIR / "endpoint_id.txt"

if "fine_tuned_model_id" not in dir():
    console.print("[yellow]Skipping — no fine-tuned model ID.[/yellow]")
else:
    hw_response = client.endpoints.list_hardware(model=fine_tuned_model_id)
    if not hw_response.data:
        raise RuntimeError("No hardware options returned for this model — check the model ID.")

    hardware = hw_response.data[0].id
    console.print(f"Using hardware: [cyan]{hardware}[/cyan]")

    endpoint = client.endpoints.create(
        model=fine_tuned_model_id,
        hardware=hardware,
        autoscaling={"min_replicas": 1, "max_replicas": 1},
        inactive_timeout=180,
        display_name="veridian-intent-eval",
    )
    # endpoint.id  — used for retrieve/update API calls (URL-safe)
    # endpoint.name — used as the model= value in chat.completions.create()
    endpoint_id   = endpoint.id
    endpoint_name = endpoint.name
    console.print(f"Endpoint created: [cyan]{endpoint_name}[/cyan] (id: {endpoint_id}) — waiting for STARTED…")

    # Poll using endpoint.id (name contains '/' which breaks the URL path)
    while True:
        ep_status = client.endpoints.retrieve(endpoint_id)
        console.print(f"  state = [yellow]{ep_status.state}[/yellow]")
        if ep_status.state == "STARTED":
            break
        if ep_status.state == "ERROR":
            raise RuntimeError("Endpoint entered error state.")
        time.sleep(20)

    # Save both: id for management calls, name for inference calls
    _ENDPOINT_ID_FILE.write_text(endpoint_id)
    _ENDPOINT_NAME_FILE.write_text(endpoint_name)
    console.print(f"[bold green]Endpoint ready.[/bold green] Saved to [cyan]{_ENDPOINT_NAME_FILE}[/cyan]")

    inference_model_id = endpoint_name

Using hardware: 2x_nvidia_h100_80gb_sxm

Endpoint created: crowe3555_2a6c/Mistral-7B-v0.1-27e98401-cdd14f64 (id: 
endpoint-98805644-a6e2-4fa3-8dcb-6ac75157704d) — waiting for STARTED…

state = PENDING

state = STARTING

state = STARTING

...

state = STARTING

state = STARTING

state = STARTING

state = STARTED

Endpoint ready. Saved to data/endpoint_name.txt

In [10]:
_ENDPOINT_NAME_FILE = DATA_DIR / "endpoint_name.txt"

if not _ENDPOINT_NAME_FILE.exists():
    console.print("[yellow]Skipping evaluation — no endpoint_name.txt found.[/yellow]\n[dim]Run the endpoint creation cell first.[/dim]")
else:
    inference_model_id = _ENDPOINT_NAME_FILE.read_text().strip()
    console.print(f"Using inference model: [cyan]{inference_model_id}[/cyan]")

    from agents.adapted_agent import _CLASSIFIER_SYSTEM_PROMPT, _INTENT_LABELS

    # ── Load test set ─────────────────────────────────────────────────────────
    test_examples: list[dict] = []
    with open(TEST_FILE) as f:
        for line in f:
            test_examples.append(json.loads(line.strip()))

    console.print(f"Loaded [bold]{len(test_examples)}[/bold] test examples")

    # ── Classify each example ─────────────────────────────────────────────────
    # Mistral-7B-v0.1 is a base model with no chat template — the chat
    # completions endpoint produces empty input. Use the text completions
    # endpoint with the Mistral instruction format applied manually.
    y_true: list[str] = []
    y_pred: list[str] = []
    rows: list[dict]  = []

    for ex in test_examples:
        msgs = ex["messages"]
        true_label  = msgs[-1]["content"]   # assistant turn = ground truth
        ticket_text = msgs[1]["content"]    # user turn = ticket text

        prompt = f"<s>[INST] {_CLASSIFIER_SYSTEM_PROMPT}\n\n{ticket_text} [/INST]"

        resp = client.completions.create(
            model=inference_model_id,
            prompt=prompt,
            temperature=0,
            max_tokens=20,
        )
        pred_label = resp.choices[0].text.strip().lower()
        if pred_label not in _INTENT_LABELS:
            pred_label = "general_question"

        y_true.append(true_label)
        y_pred.append(pred_label)
        rows.append({
            "true":      true_label,
            "predicted": pred_label,
            "correct":   true_label == pred_label,
            "text":      ticket_text[:80] + "…",
        })

    # ── Per-example table ─────────────────────────────────────────────────────
    t = Table(title="Test set predictions", header_style="bold", show_lines=False)
    t.add_column("True",      style="cyan",    min_width=20)
    t.add_column("Predicted", style="magenta", min_width=20)
    t.add_column("OK?",       justify="center")
    t.add_column("Text",      style="dim",     max_width=55)

    for row in rows:
        t.add_row(
            row["true"],
            row["predicted"],
            "[green]✓[/green]" if row["correct"] else "[red]✗[/red]",
            row["text"],
        )

    console.print(t)
    accuracy = sum(r["correct"] for r in rows) / len(rows)
    console.print(f"\nOverall accuracy: [bold]{accuracy:.0%}[/bold] ({sum(r['correct'] for r in rows)}/{len(rows)})")

Using inference model: crowe3555_2a6c/Mistral-7B-v0.1-27e98401-cdd14f64

Loaded 64 test examples

                                             Test set predictions                                              
┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ True                 ┃ Predicted            ┃ OK? ┃ Text                                                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ payments_incident    │ payments_incident    │  ✓  │ Helix is showing a critical alert for prod-payments     │
│                      │                      │     │ with a 50% error rate—need s…                           │
│ security_incident    │ security_incident    │  ✓  │ Someone installed a Slack app called 'SalesHelper Pro'  │
│                      │                      │     │ in our workspace. It has …                              │
│ general_question     │ general_question     │  ✓  │ What's the policy on working from home? I need to stay  │
│                      │                      │     │ home tomorrow but want to…                              │
│ general_question     │ general_question     │  ✓  │ Where can I find the official company holiday calendar  │
│                      │                      │     │ for next year? I need it …                              │
│ security_incident    │ hardware_issue       │  ✗  │ My laptop started making weird noises and the fan is    │
│                      │                      │     │ running nonstop—Task Manage…                            │
│ general_question     │ general_question     │  ✓  │ Is there a way to get a temporary admin password for my │
│                      │                      │     │ laptop? I need to instal…                               │
│ payments_incident    │ payments_incident    │  ✓  │ prod-payments pods are in CrashLoopBackOff across all 6 │
│                      │                      │     │ replicas in us-east-1. L…                               │
│ hardware_issue       │ software_issue       │  ✗  │ The webcam on my MacBook keeps freezing mid-call,       │
│                      │                      │     │ especially when sharing my scr…                         │
│ software_issue       │ software_issue       │  ✓  │ JetBrains IntelliJ won't activate my license—says       │
│                      │                      │     │ “connection failed” even thoug…                         │
│ payments_incident    │ payments_incident    │  ✓  │ We're seeing duplicate charge events in prod-payments — │
│                      │                      │     │ customers being charged …                               │
│ software_issue       │ payments_incident    │  ✗  │ Python package builds are failing — pip can't           │
│                      │                      │     │ authenticate to the Nexus PyPI pro…                     │
│ expense_request      │ expense_request      │  ✓  │ We have a vendor dinner in New York next Tuesday — 4    │
│                      │                      │     │ people, ~$90/person includi…                            │
│ access_request       │ access_request       │  ✓  │ I need to be added to the Okta group for the customer   │
│                      │                      │     │ support portal.…                                        │
│ access_request       │ access_request       │  ✓  │ Please remove my access to the old Nexus repo—we        │
│                      │                      │     │ migrated to the new one and I d…                        │
│ payments_incident    │ payments_incident    │  ✓  │ URGENT SEV1: prod-payments is returning HTTP 500 on all │
│                      │                      │     │ /v2/charge endpoints. Er…                               │
│ expense_request      │ expense_request      │  ✓  │ I bought a standing desk converter for my home office   │
│                      │                      │     │ and want to get reimbursed

Overall accuracy: 91% (58/64)

In [11]:
# ── sklearn classification report (precision / recall / F1 per class) ─────
if "y_true" not in dir():
    console.print("[yellow]Skipping — no evaluation results.[/yellow]")
else:
    report = classification_report(
        y_true,
        y_pred,
        labels=INTENT_LABELS,
        zero_division=0,
    )
    console.print(Panel(report, title="Classification report", border_style="green"))

╭───────────────────────────────────────────── Classification report ─────────────────────────────────────────────╮
│                    precision    recall  f1-score   support                                                      │
│                                                                                                                 │
│    access_request       0.88      1.00      0.93         7                                                      │
│ security_incident       1.00      0.88      0.93         8                                                      │
│    hardware_issue       0.89      0.89      0.89         9                                                      │
│    software_issue       0.88      0.88      0.88         8                                                      │
│        onboarding       0.78      0.88      0.82         8                                                      │
│ payments_incident       0.89      1.00      0.94         8                                                      │
│   expense_request       1.00      1.00      1.00         8                                                      │
│  general_question       1.00      0.75      0.86         8                                                      │
│                                                                                                                 │
│          accuracy                           0.91        64                                                      │
│         macro avg       0.91      0.91      0.91        64                                                      │
│      weighted avg       0.91      0.91      0.91        64                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [13]:
# ── Confusion matrix as a pandas DataFrame ────────────────────────────────
if "y_true" not in dir():
    console.print("[yellow]Skipping — no evaluation results.[/yellow]")
else:
    cm = confusion_matrix(y_true, y_pred, labels=INTENT_LABELS)
    cm_df = pd.DataFrame(cm, index=INTENT_LABELS, columns=INTENT_LABELS)
    cm_df.index.name   = "true \\ pred"

    # Style: highlight diagonal (correct predictions) in green
    def highlight_diagonal(val):
        return "background-color: #d4edda; font-weight: bold" if val > 0 else ""

    styled = (
        cm_df.style
        .map(highlight_diagonal, subset=pd.IndexSlice[INTENT_LABELS, INTENT_LABELS])
        .set_caption("Confusion matrix — rows = true label, columns = predicted label")
    )
    display(styled)  # type: ignore  # noqa: F821  (display is a Jupyter built-in)

    # Also print as plain text for non-Jupyter environments
    console.print("\n[dim]Plain text confusion matrix:[/dim]")
    console.print(cm_df.to_string())

,access_request,security_incident,hardware_issue,software_issue,onboarding,payments_incident,expense_request,general_question
true \ pred,,,,,,,,
access_request,7,0,0,0,0,0,0,0
security_incident,0,7,1,0,0,0,0,0
hardware_issue,0,0,8,1,0,0,0,0
software_issue,0,0,0,7,0,1,0,0
onboarding,1,0,0,0,7,0,0,0
payments_incident,0,0,0,0,0,8,0,0
expense_request,0,0,0,0,0,0,8,0
general_question,0,0,0,0,2,0,0,6


Plain text confusion matrix:

access_request  security_incident  hardware_issue  software_issue  onboarding  payments_incident
expense_request  general_question
true \ pred                                                                                                        
access_request                  7                  0               0               0           0                  0
0                 0
security_incident               0                  7               1               0           0                  0
0                 0
hardware_issue                  0                  0               8               1           0                  0
0                 0
software_issue                  0                  0               0               7           0                  1
0                 0
onboarding                      1                  0               0               0           7                  0
0                 0
payments_incident               0                  0               0               0           0                  8
0                 0
expense_request                 0                  0               0               0           0                  0
8                 0
general_question                0                  0               0               0           2                  0
0                 6

## 6. Save model ID

In [14]:
if "inference_model_id" not in dir():
    console.print("[yellow]Skipping — no endpoint ready to save.[/yellow]")
else:
    # classifier_model_id.txt stores the endpoint name — this is what
    # AdaptedAgent passes to client.chat.completions.create(model=...).
    # The raw fine_tuned_model_id cannot be used directly for inference.
    MODEL_ID_FILE.write_text(inference_model_id)
    console.print(f"[bold green]Classifier model ID saved →[/bold green] [cyan]{MODEL_ID_FILE}[/cyan]")
    console.print(f"\n[bold]{inference_model_id}[/bold]")
    console.print("\n[dim]Loaded automatically by 03_agent_demo.ipynb and app.py.[/dim]")

    if CONFIG_FILE.exists():
        cfg = json.loads(CONFIG_FILE.read_text())
        cfg["fine_tuned_model_id"]  = fine_tuned_model_id
        cfg["inference_model_id"]   = inference_model_id
        CONFIG_FILE.write_text(json.dumps(cfg, indent=2))
        console.print(f"[dim]Also updated {CONFIG_FILE}.[/dim]")

Classifier model ID saved → data/classifier_model_id.txt

crowe3555_2a6c/Mistral-7B-v0.1-27e98401-cdd14f64

Loaded automatically by 03_agent_demo.ipynb and app.py.

Also updated data/finetune_config.json.

## 7. Key insight

This classifier replaces the intent-routing work that `mistral-large` would otherwise
have to do during the agentic loop. A dedicated fine-tuned classifier is **faster, cheaper,
and more accurate on your specific taxonomy** than prompting a general-purpose model
to guess the intent — because it has been trained discriminatively on your exact
label set rather than relying on a large model's in-context pattern matching.

**Forge takes this further:** pre-training on your full document corpus — ticket
history, runbooks, architecture docs, internal wikis — means the generative model
itself develops *institutional fluency*, not just routing accuracy. Terms like
"Nexus", "Prism", "Helix", "prod-payments", and "SEV1" are no longer out-of-vocabulary
tokens that the model has to interpret from context; they are part of its learned
representation of the world.

The fine-tune we ran here demonstrates the supervised, label-aware variant of the same
principle — domain adaptation at the classification layer. Forge applies this at full
model scale, on-prem, with data-residency guarantees. The inference call is identical:
`client.chat.completions.create()` pointed at your Forge endpoint instead of Together.ai.

In [15]:
# ── Stop the endpoint to avoid charges ────────────────────────────────────
# Optional — the endpoint will auto-stop after 30 minutes of inactivity.
# Run this cell to stop it immediately.
if "endpoint_id" not in dir():
    console.print("[yellow]No active endpoint to stop.[/yellow]")
else:
    client.endpoints.update(endpoint_id, state="STOPPED")
    console.print(f"[bold]Endpoint [cyan]{endpoint_name}[/cyan] stopped.[/bold]")

Endpoint crowe3555_2a6c/Mistral-7B-v0.1-27e98401-cdd14f64 stopped.